# Linear SVM: hinge loss and margin regularization

Study the effect of `C` and positive-class weighting. The SVM produces margins rather than probabilities, so the cost-optimal margin threshold is learned separately.

In [ ]:
from pathlib import Path

import pandas as pd

from scania_aps.data import TEST_FILENAME, TRAIN_FILENAME, read_raw_csv

ROOT = Path.cwd().resolve()
if ROOT.name == "experiments":
    ROOT = ROOT.parent
TRAIN = ROOT / "data" / "raw" / TRAIN_FILENAME
TEST = ROOT / "data" / "raw" / TEST_FILENAME
ARTIFACTS = ROOT / "artifacts"
assert TRAIN.exists() and TEST.exists(), "Run: poetry run scania-aps download"
train = read_raw_csv(TRAIN)
test = read_raw_csv(TEST)
print(train.X.shape, test.X.shape, train.y.mean(), test.y.mean())

In [ ]:
from scania_aps.costs import optimize_score_threshold
from scania_aps.model_search import candidates_for_family
from scania_aps.models.factory import build_candidate
from scania_aps.scoring import positive_class_scores
from scania_aps.split import research_split

split = research_split(train.X, train.y)
rows = []
for candidate in candidates_for_family("linear_svm", profile="full"):
    model = build_candidate(candidate).fit(split.X_fit, split.y_fit)
    scores = positive_class_scores(model, split.X_tune)
    threshold = optimize_score_threshold(split.y_tune.to_numpy(), scores.values)
    rows.append(
        {
            **candidate.parameters,
            "threshold": threshold.threshold,
            "cost": threshold.cost.total_cost,
        }
    )
pd.DataFrame(rows).sort_values("cost")